# 02 — TYNDP 2018 web scraping

**Input:** ENTSO-E TYNDP 2018 project pages
**Output:** `2018 Webscraping/tyndp2018_project_investments.csv`, `tyndp2018_project_costs.csv`

The 2018 tabular release omits the evolution driver and delay explanation
fields, which are only available on the project web pages. This notebook
retrieves them per project.

The source pages stopped responding in February 2026. The notebook is retained
for provenance; step 03 reads the CSV files shipped in the repository.

In [9]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time, re
import os
from urllib.parse import urljoin


In [ ]:
opts = Options()
opts.add_argument("--disable-gpu")
opts.add_argument("--no-sandbox")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=opts
)

BASE = "https://tyndp-data.netlify.app"
INDEX = "/tyndp2018/projects/projects"

driver.get(BASE + INDEX)
try:
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='/tyndp2018/projects/projects/']"))
    )
    time.sleep(1)

    master_soup = BeautifulSoup(driver.page_source, "lxml")
    pattern = re.compile(r"/tyndp2018/projects/projects/\d+$")
    project_urls = {
        urljoin(BASE + INDEX, a["href"])
        for a in master_soup.find_all("a", href=True)
        if pattern.search(a["href"])
    }
    print(f"🔍 Found {len(project_urls)} projects.")
except Exception as e:
    print(f" Error retrieving project list: {e}")
    project_urls = []

investments = []
costs       = []

for url in sorted(project_urls, key=lambda u: int(u.rstrip("/").split("/")[-1])):
    try:
        driver.get(url)
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.TAG_NAME, "table"))
        )
        time.sleep(1)

        soup   = BeautifulSoup(driver.page_source, "lxml")
        tables = soup.find_all("table")

        # ——— Project Investments ———
        inv_table = None
        for tbl in tables:
            hdr = {th.get_text(strip=True).lower() for th in tbl.find_all("th")}
            if {"id", "evolution driver", "delay explanation"}.issubset(hdr):
                inv_table = tbl
                break

        if inv_table:
            inv_df = pd.read_html(str(inv_table), header=0)[0]
            cmap   = {c.lower().strip():c for c in inv_df.columns}
            inv_df = inv_df[[ cmap[k] for k in ("id","evolution driver","delay explanation") ]]
            inv_df.columns = ["investment_id","evolution_driver","delay_explanation"]
            inv_df["project_url"] = url
            investments.append(inv_df)
        else:
            print(f"Investments table not found in {url}")

        # ——— Project Cost ———
        cost_table = None
        for tbl in tables:
            hdr = {th.get_text(strip=True).lower() for th in tbl.find_all("th")}
            if {"invest nr.","capex [meuro]","uncertainty range [%]","opex [meuro/year]"}.issubset(hdr):
                cost_table = tbl
                break

        if cost_table:
            cost_df = pd.read_html(str(cost_table), header=0)[0]
            cmap = {c.lower().strip():c for c in cost_df.columns}
            # Keep only rows where 'Invest nr.' is numeric
            cost_df = cost_df[cost_df[cmap["invest nr."]].apply(lambda x: str(x).isdigit())]
            cost_df = cost_df[[ cmap[k] for k in ("invest nr.","capex [meuro]","uncertainty range [%]","opex [meuro/year]") ]]
            cost_df.columns = ["investment_id","capex_meuro","uncertainty_pct","opex_meuro_per_year"]
            cost_df["project_url"] = url
            costs.append(cost_df)
        else:
            print(f"⚠️  Cost table not found in {url}")
            
    except Exception as e:
        print(f"Error processing {url}: {e}")

driver.quit()



Starting scraping process...


In [11]:
if investments:
    df_inv = pd.concat(investments, ignore_index=True)
    df_inv.to_csv("2018 Webscraping/tyndp2018_project_investments.csv", index=False)
if costs:
    df_cost = pd.concat(costs, ignore_index=True)
    df_cost.to_csv("2018 Webscraping/tyndp2018_project_costs.csv", index=False)